In [ ]:
import tqdm as notebook_tqdm
import torch
from rich import print
from pytorch_tabular.utils import load_covertype_dataset
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device == "cuda":
    #torch.set_float32_matmul_precision('medium' | 'high')
    torch.set_float32_matmul_precision('high')

In [ ]:
#data, cat_col_names, num_col_names, target_col = load_covertype_dataset()

model, data = final_extinctionrisk_dataframe()
categorical_data = data.drop(model.numeric, axis=1)
cat_col_names = list(categorical_data.columns)
num_col_names = model.numeric

print(f"Data Shape: {data.shape} | # of cat cols: {len(cat_col_names)} | # of num cols: {len(num_col_names)}")
print(f"[bold dodger_blue2] Features: {num_col_names + cat_col_names}[/bold dodger_blue2]")
print(f"[bold purple4]Target: {model.label}[/bold purple4]")

In [ ]:
# Let's also check the data for missing values
print(data.isna().sum())

In [ ]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(data, random_state=42, test_size=0.2)
train, val = train_test_split(train, random_state=42, test_size=0.2)
print(f"Train Shape: {train.shape} | Val Shape: {val.shape} | Test Shape: {test.shape}")

In [ ]:
from pytorch_tabular.models import GANDALFConfig, TabNetModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

data_config = DataConfig(
    target=[
        model.label
    ],  # target should always be a list
    continuous_cols=num_col_names,
    categorical_cols=cat_col_names,
    num_workers = 0, 
    pin_memory=True
)
trainer_config = TrainerConfig(
    batch_size=128,
    max_epochs=50,
)
optimizer_config = OptimizerConfig()

model_config = TabNetModelConfig(
    n_d = 8,
    n_a = 8,
    n_steps = 3,
    gamma = 1.3,
    n_independent = 2,
    n_shared = 2,
    virtual_batch_size=128,
    mask_type = 'sparsemax',
    task="classification",
    head = 'LinearHead',
    embedding_dropout = 0.0,
    batch_norm_continuous_input = True,
    learning_rate = 0.001,
    seed = 42,
)

In [ ]:
from pytorch_tabular import TabularModel

tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
    verbose=True
)

In [ ]:
tabular_model.fit(train=train, validation=val)

In [ ]:
pred_df = tabular_model.predict(test)
pred_df.head()

In [ ]:
result = tabular_model.evaluate(test)

In [ ]:
tabular_model.save_model("examples/basic")

In [ ]:
loaded_model = TabularModel.load_model("examples/basic")

In [ ]:
result = loaded_model.evaluate(test)